# 👕 Virtual Try-On — Google Colab Setup

**Bilkul free. Koi payment nahi. Bas cells upar se neeche run karo.**

---

### Pehle Yeh Karo (Ek Baar):
1. Menu → **Runtime** → **Change runtime type**
2. Hardware accelerator = **T4 GPU** select karo
3. **Save** karo
4. Phir neeche cells run karo

---
| Cell | Kya karta hai | Time |
|------|--------------|------|
| 1 | GPU check | 5 sec |
| 2 | PyTorch + CUDA verify | 10 sec |
| 3 | Detectron2 install | 3-5 min |
| 4 | Project clone | 1-2 min |
| 5 | Requirements install | 3-5 min |
| 6 | DensePose model download | 1-2 min |
| 7 | App launch (public URL milega) | 15-20 min (model load) |

In [ ]:
# ── Cell 1: GPU Check ─────────────────────────────────────────────────────────
import torch

print('='*50)
print('GPU Available:', torch.cuda.is_available())

if torch.cuda.is_available():
    print('GPU Name   :', torch.cuda.get_device_name(0))
    vram_gb = round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1)
    print('VRAM       :', vram_gb, 'GB')
    print('CUDA Ver   :', torch.version.cuda)
    print('PyTorch Ver:', torch.__version__)
    print('='*50)
    if vram_gb < 12:
        print('⚠️  WARNING: 12GB+ VRAM chahiye. T4 (15GB) use karo.')
    else:
        print('✅ GPU ready hai!')
else:
    print('='*50)
    print('❌ GPU nahi mila!')
    print('   Runtime > Change runtime type > T4 GPU select karo')
    raise SystemExit('GPU required — runtime type change karo phir restart karo')

In [ ]:
# ── Cell 2: Detectron2 Install (special — pip se direct nahi hota) ────────────
# Yeh automatically sahi version detect kar ke install karta hai

import torch
import subprocess

cuda_ver = torch.version.cuda.replace('.', '')[:3]   # e.g. '121'
torch_ver = torch.__version__.split('+')[0]           # e.g. '2.3.0'
torch_major_minor = 'torch' + '.'.join(torch_ver.split('.')[:2])  # e.g. 'torch2.3'

wheel_url = f'https://dl.fbaipublicfiles.com/detectron2/wheels/cu{cuda_ver}/{torch_major_minor}/index.html'

print(f'CUDA: {cuda_ver} | PyTorch: {torch_ver}')
print(f'Wheel URL: {wheel_url}')
print('Detectron2 install ho raha hai...')

# Pehle pre-built wheel try karo (fast)
result = subprocess.run(
    ['pip', 'install', 'detectron2', '-f', wheel_url, '-q'],
    capture_output=True, text=True
)

if result.returncode != 0:
    # Pre-built wheel nahi mila — source se build karo (slow but works)
    print('Pre-built wheel nahi mila. Source se build ho raha hai (5-10 min)...')
    subprocess.run(
        ['pip', 'install', 'git+https://github.com/facebookresearch/detectron2.git', '-q'],
        check=True
    )

# Verify
import detectron2
print(f'✅ Detectron2 {detectron2.__version__} install ho gaya!')

In [ ]:
# ── Cell 3: Project Clone from GitHub ────────────────────────────────────────
import os, subprocess

GITHUB_USERNAME = 'zaryabali786'
REPO_NAME       = 'try-your-self'
PROJECT_DIR     = '/content/try-your-self'

GITHUB_URL = f'https://github.com/{GITHUB_USERNAME}/{REPO_NAME}'

os.chdir('/content')

if os.path.isdir(PROJECT_DIR):
    print(f'✅ Project already hai: {PROJECT_DIR}')
else:
    print(f'Cloning from: {GITHUB_URL}')
    result = subprocess.run(['git', 'clone', GITHUB_URL, PROJECT_DIR])
    if result.returncode == 0:
        print('✅ Clone complete!')
    else:
        raise RuntimeError('❌ Clone fail hua. GitHub repo public hai?')

os.chdir(PROJECT_DIR)
print(f'Working directory: {os.getcwd()}')
print('Files:', os.listdir('.'))

In [ ]:
# ── Cell 4: Requirements Install ─────────────────────────────────────────────
# ⚠️  ZAROORI: Yeh cell chalao Cell 7 se pehle — warna numpy/scipy crash karega

print('[1/5] diffusers aur gradio remove kar raha hai...')
!pip uninstall -y diffusers gradio 2>/dev/null; echo "done"

print()
print('[2/5] diffusers 0.20.2 install (--no-deps)...')
!pip install -q --no-deps "diffusers==0.20.2"
!pip install -q "safetensors>=0.3.1" "omegaconf"

print()
print('[3/5] constants.py file patch (hf_cache_home)...')
import os, glob

for path in glob.glob('/usr/local/lib/python*/dist-packages/diffusers/utils/constants.py'):
    with open(path) as f: src = f.read()
    OLD = 'from huggingface_hub.constants import HUGGINGFACE_HUB_CACHE, hf_cache_home'
    NEW = ('from huggingface_hub.constants import HUGGINGFACE_HUB_CACHE\n'
           'hf_cache_home = os.path.expanduser(\n'
           '    os.getenv("HF_HOME", os.path.join(os.getenv("XDG_CACHE_HOME","~/.cache"),"huggingface"))\n'
           ')')
    if OLD in src:
        with open(path, 'w') as f: f.write(src.replace(OLD, NEW))
        print(f'    ✅ Patched')
    else:
        print(f'    ℹ️  Already ok')

print()
print('[4/5] gradio + libraries install...')
# gradio --no-deps: numpy downgrade se bachao (scipy ke liye numpy 2.x zaroori hai)
!pip install -q --no-deps "gradio==3.41.2"
!pip install -q "aiofiles" "aiohttp" "altair" "fastapi" "ffmpy" "httpx" \
    "huggingface-hub" "jinja2" "markdown-it-py" "orjson" "pandas" \
    "pillow" "pydantic" "python-multipart" "pyyaml" "semantic-version" \
    "typing-extensions" "uvicorn[standard]" "websockets" 2>/dev/null; echo "gradio deps done"

# Image + model libraries (numpy version mat badlna — Colab ka 2.x rakho)
!pip install -q Pillow scipy scikit-image opencv-python
!pip install -q einops onnxruntime cloudpickle
!pip install -q fvcore iopath pycocotools termcolor tabulate psutil
!pip install -q matplotlib tqdm basicsr
!pip install -q fastapi "uvicorn[standard]" python-multipart

print()
print('[5/5] monkey_patch verify...')
import sys
sys.path.insert(0, '/content/try-your-self')
import monkey_patch
print('    ✅ monkey_patch OK')

# numpy version check
import numpy as np
print(f'    numpy: {np.__version__} (2.x hona chahiye)')
if int(np.__version__.split('.')[0]) < 2:
    print('    ⚠️  numpy 1.x hai — scipy crash ho sakta hai!')
    print('       Chalao: !pip install "numpy>=2.0" phir Runtime Restart karo')
else:
    print('    ✅ numpy version OK')

print()
print('✅ Cell 4 complete!')

In [ ]:
# ── Cell 5: DensePose Model Download ─────────────────────────────────────────
import os, subprocess

os.chdir('/content/try-your-self')

ckpt_dir  = '/content/try-your-self/ckpt/densepose'
ckpt_file = os.path.join(ckpt_dir, 'model_final_162be9.pkl')

if os.path.exists(ckpt_file):
    size_mb = os.path.getsize(ckpt_file) / 1e6
    print(f'✅ DensePose model already hai ({size_mb:.0f} MB) — skip')
else:
    os.makedirs(ckpt_dir, exist_ok=True)
    print('DensePose model download ho raha hai (~250 MB)...')
    subprocess.run([
        'wget', '-q', '--show-progress',
        'https://dl.fbaipublicfiles.com/detectron2/densepose/densepose_rcnn_R_50_FPN_s1x/165712039/model_final_162be9.pkl',
        '-O', ckpt_file
    ], check=True)
    print(f'✅ Download complete!')

In [ ]:
# ── Cell 6: Final Verify ─────────────────────────────────────────────────────
import os

os.chdir('/content/try-your-self')

checks = {
    'app.py'                                   : os.path.exists('app.py'),
    'api.py'                                   : os.path.exists('api.py'),
    'configs/densepose_rcnn_R_50_FPN_s1x.yaml' : os.path.exists('configs/densepose_rcnn_R_50_FPN_s1x.yaml'),
    'ckpt/densepose/model_final_162be9.pkl'    : os.path.exists('ckpt/densepose/model_final_162be9.pkl'),
    'src/tryon_pipeline.py'                    : os.path.exists('src/tryon_pipeline.py'),
    'preprocess/'                              : os.path.exists('preprocess'),
}

all_ok = True
print('=== Pre-flight Check ===')
for name, ok in checks.items():
    status = '✅' if ok else '❌'
    print(f'  {status}  {name}')
    if not ok:
        all_ok = False

print()
if all_ok:
    print('✅ Sab ready hai! Cell 7 run karo.')
else:
    print('❌ Kuch missing hai — upar ke cells dobara check karo.')

---
## App Launch Karo

**Cell 7** = Gradio Web UI (browser mein use karo)  
**Cell 8** = FastAPI (Postman / mobile app se test karo)  

Dono mein se ek hi chalao — dono ek saath nahi.

In [ ]:
# ── Cell 7: Direct Inference — No Gradio, No ngrok ───────────────────────────
# Bas URLs dalo, result seedha Colab mein dikhe ga
# ═══════════════════════════════════════════════════════════════════════════════

import os
os.chdir('/content/try-your-self')

# ── INPUT — Yahan apni images ka URL dalo ─────────────────────────────────────
HUMAN_IMAGE_URL   = "https://images.unsplash.com/photo-1529139574466-a303027c1d8b?w=768"  # ← apna URL
GARMENT_IMAGE_URL = "https://images.unsplash.com/photo-1503341504253-dff4815485f1?w=768"  # ← apna URL
GARMENT_DESC      = "white t-shirt"   # ← kapray ki description

# Advanced settings (change karna zaroori nahi)
USE_AUTO_MASK  = True
USE_AUTO_CROP  = False
DENOISE_STEPS  = 30
SEED           = 42
# ═══════════════════════════════════════════════════════════════════════════════

import sys, requests
from PIL import Image
from io import BytesIO
from IPython.display import display, Image as IPImage
import torch

# monkey_patch pehle (compatibility fixes)
import monkey_patch

print("Images download ho rahi hain...")

def load_image_from_url(url):
    response = requests.get(url, timeout=30)
    response.raise_for_status()
    return Image.open(BytesIO(response.content)).convert("RGB")

human_img   = load_image_from_url(HUMAN_IMAGE_URL)
garment_img = load_image_from_url(GARMENT_IMAGE_URL)
print(f"  Human image  : {human_img.size}")
print(f"  Garment image: {garment_img.size}")

print("\nModels load ho rahe hain (pehli baar ~15-20 min)...")

from src.tryon_pipeline import StableDiffusionXLInpaintPipeline as TryonPipeline
from src.unet_hacked_garmnet import UNet2DConditionModel as UNet2DConditionModel_ref
from src.unet_hacked_tryon import UNet2DConditionModel
from transformers import CLIPImageProcessor, CLIPVisionModelWithProjection, CLIPTextModel, CLIPTextModelWithProjection, AutoTokenizer
from diffusers import DDPMScheduler, AutoencoderKL
from torchvision import transforms
from torchvision.transforms.functional import to_pil_image
from contextlib import nullcontext
import numpy as np
import apply_net
from preprocess.humanparsing.run_parsing import Parsing
from preprocess.openpose.run_openpose import OpenPose
from detectron2.data.detection_utils import convert_PIL_to_numpy, _apply_exif_orientation
from utils_mask import get_mask_location

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE  = torch.float16 if DEVICE == "cuda" else torch.float32
BASE   = 'yisol/IDM-VTON'

unet         = UNet2DConditionModel.from_pretrained(BASE, subfolder="unet", torch_dtype=DTYPE)
tok1         = AutoTokenizer.from_pretrained(BASE, subfolder="tokenizer", use_fast=False)
tok2         = AutoTokenizer.from_pretrained(BASE, subfolder="tokenizer_2", use_fast=False)
scheduler    = DDPMScheduler.from_pretrained(BASE, subfolder="scheduler")
enc1         = CLIPTextModel.from_pretrained(BASE, subfolder="text_encoder", torch_dtype=DTYPE)
enc2         = CLIPTextModelWithProjection.from_pretrained(BASE, subfolder="text_encoder_2", torch_dtype=DTYPE)
img_enc      = CLIPVisionModelWithProjection.from_pretrained(BASE, subfolder="image_encoder", torch_dtype=DTYPE)
vae          = AutoencoderKL.from_pretrained(BASE, subfolder="vae", torch_dtype=DTYPE)
unet_encoder = UNet2DConditionModel_ref.from_pretrained(BASE, subfolder="unet_encoder", torch_dtype=DTYPE)

for m in [unet, enc1, enc2, img_enc, vae, unet_encoder]:
    m.requires_grad_(False)

parsing_model  = Parsing(0)
openpose_model = OpenPose(0)

pipe = TryonPipeline.from_pretrained(
    BASE, unet=unet, vae=vae,
    feature_extractor=CLIPImageProcessor(),
    text_encoder=enc1, text_encoder_2=enc2,
    tokenizer=tok1, tokenizer_2=tok2,
    scheduler=scheduler, image_encoder=img_enc,
    torch_dtype=DTYPE,
)
pipe.unet_encoder = unet_encoder

print("\nInference chal rahi hai...")

openpose_model.preprocessor.body_estimation.model.to(DEVICE)
pipe.to(DEVICE)
pipe.unet_encoder.to(DEVICE)

tensor_transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize([0.5],[0.5])])

garm  = garment_img.resize((768,1024))
human = human_img.resize((768,1024))

keypoints   = openpose_model(human.resize((384,512)))
model_parse,_ = parsing_model(human.resize((384,512)))
mask, _     = get_mask_location('hd', "upper_body", model_parse, keypoints)
mask        = mask.resize((768,1024))

mask_gray   = (1 - transforms.ToTensor()(mask)) * tensor_transform(human)
mask_gray   = to_pil_image((mask_gray + 1.0) / 2.0)

human_arg   = _apply_exif_orientation(human.resize((384,512)))
human_arg   = convert_PIL_to_numpy(human_arg, format="BGR")

args = apply_net.create_argument_parser().parse_args((
    'show','./configs/densepose_rcnn_R_50_FPN_s1x.yaml',
    './ckpt/densepose/model_final_162be9.pkl',
    'dp_segm','-v','--opts','MODEL.DEVICE', DEVICE
))
pose_img = Image.fromarray(args.func(args, human_arg)[:,:,::-1]).resize((768,1024))

with torch.no_grad():
    with (torch.cuda.amp.autocast() if DEVICE=="cuda" else nullcontext()):
        prompt1 = "model is wearing " + GARMENT_DESC
        prompt2 = "a photo of " + GARMENT_DESC
        neg     = "monochrome, lowres, bad anatomy, worst quality, low quality"

        with torch.inference_mode():
            pe,npe,ppe,nppe = pipe.encode_prompt(prompt1,num_images_per_prompt=1,do_classifier_free_guidance=True,negative_prompt=neg)
        with torch.inference_mode():
            pec,_,_,_ = pipe.encode_prompt([prompt2],num_images_per_prompt=1,do_classifier_free_guidance=False,negative_prompt=[neg])

        pose_t = tensor_transform(pose_img).unsqueeze(0).to(DEVICE,DTYPE)
        garm_t = tensor_transform(garm).unsqueeze(0).to(DEVICE,DTYPE)
        gen    = torch.Generator(DEVICE).manual_seed(SEED)

        result = pipe(
            prompt_embeds=pe.to(DEVICE,DTYPE), negative_prompt_embeds=npe.to(DEVICE,DTYPE),
            pooled_prompt_embeds=ppe.to(DEVICE,DTYPE), negative_pooled_prompt_embeds=nppe.to(DEVICE,DTYPE),
            num_inference_steps=DENOISE_STEPS, generator=gen, strength=1.0,
            pose_img=pose_t, text_embeds_cloth=pec.to(DEVICE,DTYPE),
            cloth=garm_t, mask_image=mask, image=human,
            height=1024, width=768, ip_adapter_image=garm.resize((768,1024)),
            guidance_scale=2.0,
        )[0]

output_img = result[0]
output_img.save("/content/tryon_result.png")

print("\n✅ Done! Result:")
print("Saved: /content/tryon_result.png")
print()

# Colab mein seedha dikhao
from IPython.display import display
print("--- Input Images ---")
display(human_img.resize((300,400)), garment_img.resize((300,400)))
print("--- Try-On Result ---")
display(output_img.resize((300,400)))

In [ ]:
# ── Cell 8: FastAPI + ngrok Public URL ───────────────────────────────────────
import os, threading, time
os.chdir('/content/try-your-self')

!pip install -q pyngrok

from pyngrok import ngrok, conf

# ngrok auth token
ngrok.set_auth_token("cr_3CwBSXiGbcKrxhiM4DrnLMbqkmN")

# API server background mein start karo
def start_api():
    os.system('cd /content/try-your-self && python api.py')

thread = threading.Thread(target=start_api, daemon=True)
thread.start()

print('API server start ho raha hai...')
time.sleep(10)

# Public tunnel
public_url = ngrok.connect(8000)
print()
print('='*60)
print(f'✅ API Public URL: {public_url}')
print('='*60)
print()
print('Postman mein use karo:')
print(f'  GET  {public_url}/health')
print(f'  POST {public_url}/tryon')
print(f'  POST {public_url}/tryon/both')
print()
print('Swagger docs: ' + str(public_url) + '/docs')

---
## Masail / Troubleshooting

| Masla | Hall |
|-------|------|
| `CUDA out of memory` | Runtime > Restart, phir Cell 7 directly run karo |
| `No module named detectron2` | Cell 2 dobara run karo |
| `model_final_162be9.pkl not found` | Cell 5 dobara run karo |
| URL nahi aa raha | 2-3 min wait karo, Cell 7 ki output scroll karo |
| Colab disconnect ho gaya | Sab cells fresh run karo (model cache rehta hai Drive mein nahi, phir download hoga) |

---
**Note:** Colab free tier mein session ~12 ghante chalta hai. Baad mein phir se run karna parega.